# Week 2 Day 5: choosing the policy wording

A standalone adaptation of the self-improvement lab's assignment.

The claim agent's configuration gains a fourth choice: `full` or `short` policy wording. The improver starts by testing this choice alone, then can refine its recipe. Human approval and the original quality, noise and rollback rules remain in place. Model weights are unchanged.

In [ ]:
import difflib
import json
import operator
import os
import sys
from datetime import datetime
from pathlib import Path
from typing import Annotated, Literal, TypedDict

import pandas as pd
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.state import CompiledStateGraph
from langgraph.types import Command, RetryPolicy, Send
from openrouter import OpenRouter
from pydantic import BaseModel, ValidationError

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "labs/week2day5.ipynb").is_file()
)
sys.path.insert(0, str(ROOT / "labs"))
from week2day1_payout import Benefit, Line, Payout, work_out_payout

## Data and versioned configuration

The evaluation code reads the fictional handler labels. The deciding agent receives only the claim, policy record, tools and selected wording. The improver receives evaluation examples and aggregate held-out scores, never held-out examples.

In [ ]:
DATA = ROOT / "data"


def load(name: str) -> list | dict:
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny: bool = False, held_out: bool = False) -> tuple[list, dict]:
    """Return a list of claims, and the handlers' decisions on them keyed by claim id."""
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load(
        "evals/held_out_decisions.json" if held_out else "evals/decisions.json"
    )
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [ ]:
TINY = False

evaluation_claims, evaluation_labels = get_sample_data(tiny=TINY)
held_out_claims, held_out_labels = get_sample_data(tiny=TINY, held_out=True)

claims = {claim["claim_id"]: claim for claim in evaluation_claims + held_out_claims}
labels = evaluation_labels | held_out_labels
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
WORDINGS = {
    "full": (DATA / "policy_wording.md").read_text(encoding="utf-8"),
    "short": (DATA / "policy_wording_short.md").read_text(encoding="utf-8"),
}

SETS = {"evaluation": list(evaluation_labels), "held-out": list(held_out_labels)}
print(
    f"{len(SETS['evaluation'])} evaluation claims and {len(SETS['held-out'])} held-out claims"
)

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using the payment section.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.

Explain your decision in 2-3 sentences, citing the relevant section title or number. Do not invent section numbers.
Use your tools to record your decision.
Use the payout tool to work out the payout on an approved claim, and record the payout it returns.
"""


class Configuration(BaseModel):
    version: int
    instructions: str
    model: str
    reasoning_effort: str
    policy_wording: Literal["full", "short"]
    reason: str = "Baseline"


versions = [
    Configuration(
        version=1,
        instructions=INSTRUCTIONS,
        model="openai/gpt-6-luna",
        reasoning_effort="low",
        policy_wording="full",
    )
]
current = 1


def claim_system_prompt(config: Configuration) -> str:
    """Use exactly the wording saved with this version."""
    return f"{config.instructions}\n{WORDINGS[config.policy_wording]}\n"

## Claim agent and dispatcher

These retain the lab's payout tool and parallel claim workers. All decisions stay in notebook memory.

In [ ]:
recorded = {}


@tool(parse_docstring=True)
def record_decision(
    claim_id: str,
    rationale: str,
    decision: Literal["approve", "decline", "refer"],
    payout: float,
) -> str:
    """Record the decision on a claim in the claims system.

    Args:
        claim_id: The claim's id
        rationale: 2-3 sentences to explain the decision along with the relevant section of the policy
        decision: The decision on the claim
        payout: The amount to pay in US dollars, or 0 for a decline or a referral
    """
    if claim_id in recorded:
        return f"A decision is already recorded on {claim_id}, and decisions are final"
    if decision == "approve" and payout <= 0:
        return "An approval needs a payout above zero"
    recorded[claim_id] = {
        "decision": decision,
        "payout": payout,
        "rationale": rationale,
    }
    return f"Recorded {decision} on {claim_id}"


@tool(parse_docstring=True)
def calculate_payout(
    policy_id: str, benefit: Benefit, lines: list[Line], paid_by_others: float = 0.0
) -> Payout:
    """Work out the payout on an approved claim with UnderCover's payout rules.

    Args:
        policy_id: The customer's policy id
        benefit: The benefit the claim is paid under
        lines: The covered items and expenses, one for each line of the claim, with any excluded ones left out. A pair or a set counts as one item
        paid_by_others: Anything a carrier, hotel or other party has paid or refunded for the same loss

    Returns:
        The calculation, the payout and whether it needs sign-off
    """
    return work_out_payout(policies[policy_id], benefit, lines, paid_by_others)

In [ ]:
def make_user_prompt(claim: dict) -> str:
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""


def make_agent(config: Configuration) -> CompiledStateGraph:
    """Build the claim agent for one version of the configuration."""
    model = ChatOpenAI(
        model=config.model,
        base_url=OPENROUTER_BASE_URL,
        api_key=openrouter_api_key,
        reasoning_effort=config.reasoning_effort,
        timeout=120,
        max_retries=5,
    )
    return create_agent(
        model=model,
        tools=[record_decision, calculate_payout],
        system_prompt=claim_system_prompt(config),
    )

In [ ]:
class Dispatch(TypedDict):
    """What the dispatcher keeps for one run."""

    config: Configuration
    claims: list[str]
    results: Annotated[list, operator.add]


def fan_out(state: Dispatch) -> list[Send]:
    """Send each claim to a claim agent, with the configuration to use."""
    return [
        Send("work_claim", {"claim_id": claim_id, "config": state["config"]})
        for claim_id in state["claims"]
    ]


def work_claim(task: dict) -> dict:
    """Run a claim agent on one claim and return its result."""
    claim_id = task["claim_id"]
    agent = make_agent(task["config"])
    started = datetime.now()
    result = agent.invoke(
        {"messages": [{"role": "user", "content": make_user_prompt(claims[claim_id])}]}
    )
    answer = recorded.get(
        claim_id, {"decision": "refer", "payout": 0.0, "rationale": "Nothing recorded"}
    )
    cost = sum(
        message.response_metadata["token_usage"]["cost"]
        for message in result["messages"]
        if message.type == "ai"
    )
    return {
        "results": [
            {
                "claim_id": claim_id,
                **answer,
                "cost": cost,
                "started": started,
                "finished": datetime.now(),
            }
        ]
    }


builder = StateGraph(Dispatch)
builder.add_node("work_claim", work_claim, retry_policy=RetryPolicy(max_attempts=3))
builder.add_conditional_edges(START, fan_out, ["work_claim"])
builder.add_edge("work_claim", END)
dispatcher = builder.compile()

## Judge and evaluation rules

Jev uses the lab's reference-guided F/C/B/A rubric (0 to 3). Both wordings use the same judge. Equivalent section titles and numbers count as the same reference, since the short file has no numbered headings.

Keep a candidate only if held-out accuracy is at least 0.60 and grade at least 1.0, one measure improves beyond noise, and none worsens beyond tolerance. Cost must fall by more than 15% to count as an improvement. The held-out set selects versions; it is therefore a validation set, not an independent final test.

In [ ]:
JEV = "typesafe/jev-1.13"

# Jev numbers the grades by their position in the list, so F is 0 and A is 3
RUBRIC = {
    "F": "different grounds from the handler's reason, or it contradicts the handler's reason",
    "C": "partly the same as the handler's reason: one of the handler's grounds or the section is missing, or it adds a ground the handler did not give",
    "B": "the same grounds and the same section of the policy wording as the handler's reason",
    "A": "the same grounds and section as the handler's reason, and clearly better expressed: clearer for the customer, with the section and the figures, shorter, and no jargon",
}
GRADES = list(RUBRIC)

JUDGE_QUESTIONS = {
    "grade": {
        "type": "score",
        "instructions": "An expert claims handler decided a travel insurance claim and gave `handler.reason`, resting on `handler.reference`. Grade how well `agent.rationale` explains the same decision.",
        "criteria": [f"{letter}: {text}" for letter, text in RUBRIC.items()],
    },
    "same_grounds": {
        "type": "noul",
        "instructions": "`agent.rationale` gives the same main grounds for the decision as `handler.reason`",
    },
    "same_section": {
        "type": "noul",
        "instructions": "`agent.rationale` refers to the same section of the policy wording as `handler.reference`",
    },
}

SECTION_EQUIVALENTS = (
    "Equivalent section names/numbers: Limits=1; Cancellation and Interruption=2; "
    "Trip Delay=3; Baggage Delay=4; Baggage=5; Passport=6; Emergency Medical=7; "
    "Add-ons=8; Never covered=9; Payment=10; Claims=11 and Definitions=12. "
    "Equivalent titles and numbers count as the same section when grading."
)
for question in ("grade", "same_section"):
    JUDGE_QUESTIONS[question]["instructions"] += " " + SECTION_EQUIVALENTS


def judge(rationale: str, decision: str, expected: dict) -> dict:
    """Ask Jev to grade a rationale against the handler's reason."""
    state = {
        "handler": {
            "decision": expected["decision"],
            "reason": expected["reason"],
            "reference": expected["reference"],
        },
        "agent": {"decision": decision, "rationale": rationale},
    }
    response = openrouter_decisions.alpha.decisions.create(
        model=JEV, state=state, questions=JUDGE_QUESTIONS
    )
    answers = response.answers
    probabilities = answers["grade"].probabilities
    most_likely = max(probabilities, key=probabilities.get)
    return {
        "grade": answers["grade"].score,
        "letter": GRADES[int(most_likely)],
        "same_grounds": answers["same_grounds"].noul,
        "same_section": answers["same_section"].noul,
        "judge_cost": response.usage.cost,
    }

In [ ]:
WRONGFUL_DECLINE = 1500
NEEDLESS_REFERRAL = 50


def is_right(answer: dict, expected: dict) -> bool:
    """True when the decision matches the handler's and the payout is within a dollar."""
    return (
        answer["decision"] == expected["decision"]
        and abs(answer["payout"] - expected["payout"]) <= 1
    )


def mistake_cost(answer: dict, expected: dict) -> float:
    """What the agent's answer costs UnderCover, against the handler's decision."""
    if answer["decision"] == expected["decision"]:
        return (
            abs(answer["payout"] - expected["payout"])
            if answer["decision"] == "approve"
            else 0
        )
    if answer["decision"] == "decline":
        return WRONGFUL_DECLINE
    if answer["decision"] == "approve":
        return answer["payout"]
    return NEEDLESS_REFERRAL

In [ ]:
def score_table(results: list[dict]) -> pd.DataFrame:
    """One row for each claim: the agent's answer beside the handler's, with the judge's grade, cost and time."""
    rows = []
    for answer in results:
        expected = labels[answer["claim_id"]]
        rows.append(
            {
                "claim_id": answer["claim_id"],
                "handler": expected["decision"],
                "agent": answer["decision"],
                "handler_payout": expected["payout"],
                "agent_payout": answer["payout"],
                "right": is_right(answer, expected),
                "mistake_cost": mistake_cost(answer, expected),
                **judge(answer["rationale"], answer["decision"], expected),
                "llm_cost": answer["cost"],
                "seconds": (answer["finished"] - answer["started"]).total_seconds(),
                "rationale": answer["rationale"],
            }
        )
    return pd.DataFrame(rows).sort_values("claim_id")


def run_scores(table: pd.DataFrame) -> dict:
    """The scores for a whole run: quality, cost and time."""
    return {
        "decision_accuracy": (table["agent"] == table["handler"]).mean(),
        "accuracy": table["right"].mean(),
        "cost_of_mistakes": table["mistake_cost"].sum(),
        "grade": table["grade"].mean(),
        "cost_per_claim": table["llm_cost"].mean(),
        "seconds_per_claim": table["seconds"].mean(),
    }

In [ ]:
tables = {}
scoreboard = []
spent = 0.0


def evaluate(config: Configuration, set_name: str) -> dict:
    """Run one version of the configuration on one set, score it and add it to the scoreboard."""
    global spent
    recorded.clear()
    run_name = f"Version {config.version} on the {set_name} set"
    result = dispatcher.invoke(
        {"config": config, "claims": SETS[set_name]},
        {"max_concurrency": 25, "callbacks": [langfuse_handler], "run_name": run_name},
    )
    table = score_table(result["results"])
    scores = run_scores(table)
    spent += table["llm_cost"].sum() + table["judge_cost"].sum()
    tables[(config.version, set_name)] = table
    scoreboard.append(
        {
            "version": config.version,
            "set": set_name,
            "wording": config.policy_wording,
            **scores,
        }
    )
    print(
        f"{run_name}: accuracy {scores['accuracy']:.2f}, grade {scores['grade']:.2f}, "
        f"${scores['cost_per_claim']:.5f} and {scores['seconds_per_claim']:.1f} seconds a claim"
    )
    return scores

In [ ]:
OBJECTIVES = {"accuracy": 0.6, "grade": 1.0}
NOISE = {
    "accuracy": 0.1,
    "grade": 0.15,
    "cost_per_claim": 0.15,
    "seconds_per_claim": 0.15,
}
TOLERANCE = {
    "accuracy": 0.1,
    "grade": 0.15,
    "cost_per_claim": 1.0,
    "seconds_per_claim": 0.25,
}
BUDGET = 0.50

ALLOWED_MODELS = {
    "openai/gpt-6-luna": {
        "reasoning_efforts": ["none", "low", "medium"],
        "price per million tokens": "$0.10 in, $0.50 out",
    },
    "google/gemini-3.5-flash-lite": {
        "reasoning_efforts": ["minimal", "low"],
        "price per million tokens": "$0.30 in, $2.50 out",
    },
    "deepseek/deepseek-v4.1-flash": {
        "reasoning_efforts": ["low"],
        "price per million tokens": "$0.30 in, $1.20 out",
    },
}

In [ ]:
def compare(new: dict, old: dict) -> tuple[list[str], list[str]]:
    """Which scores got better by more than the noise, and which got worse by more than the tolerance."""
    better, worse = [], []
    for name in ["accuracy", "grade"]:
        if new[name] > old[name] + NOISE[name]:
            better.append(name)
        if new[name] < old[name] - TOLERANCE[name]:
            worse.append(name)
    for name in ["cost_per_claim", "seconds_per_claim"]:
        if new[name] < old[name] * (1 - NOISE[name]):
            better.append(name)
        if new[name] > old[name] * (1 + TOLERANCE[name]):
            worse.append(name)
    return better, worse


def keep_or_roll_back(new: dict, old: dict) -> tuple[bool, str]:
    """Decide whether to keep a new version from its held-out scores and the current version's average."""
    missed = [name for name, floor in OBJECTIVES.items() if new[name] < floor]
    better, worse = compare(new, old)
    if missed:
        return False, f"it missed the objectives on {', '.join(missed)}"
    if worse:
        return False, f"{', '.join(worse)} got worse"
    if not better:
        return False, "no score improved by more than the noise"
    return True, f"{', '.join(better)} improved"

## Improver tools

Versions store the selected wording; reports, proposals and round history expose it. The original allowlist, evaluation budget guard and approval pauses remain in place.

In [ ]:
rounds = []


def latest(version: int, set_name: str) -> dict:
    """The most recent scores for a version on a set."""
    return [
        row
        for row in scoreboard
        if row["version"] == version and row["set"] == set_name
    ][-1]


def average(version: int, set_name: str) -> dict:
    """The average of a version's scores over all its runs on a set."""
    runs = pd.DataFrame(
        [
            row
            for row in scoreboard
            if row["version"] == version and row["set"] == set_name
        ]
    )
    return runs.drop(columns=["version", "set", "wording"]).mean().to_dict()


def beside_handler(table: pd.DataFrame) -> list[dict]:
    """Each claim in a table with the agent's answer beside the handler's and the judge's grade."""
    rows = []
    for row in table.to_dict("records"):
        expected = labels[row["claim_id"]]
        rows.append(
            {
                "claim_id": row["claim_id"],
                "summary": claims[row["claim_id"]]["summary"],
                "agent": {
                    "decision": row["agent"],
                    "payout": row["agent_payout"],
                    "rationale": row["rationale"],
                },
                "handler": {
                    "decision": expected["decision"],
                    "payout": expected["payout"],
                    "reason": expected["reason"],
                    "reference": expected["reference"],
                },
                "judge": {"grade": round(row["grade"], 2), "letter": row["letter"]},
            }
        )
    return rows


def mistakes(version: int) -> dict:
    """The evaluation claims a version got wrong, and the five right ones with the lowest grades."""
    table = tables[(version, "evaluation")]
    weakest = table[table["right"]].sort_values("grade").head(5)
    return {
        "wrong": beside_handler(table[~table["right"]]),
        "explained_least_well": beside_handler(weakest),
    }

In [ ]:
@tool
def read_results() -> str:
    """Read the current version of the claim agent's configuration, its scores, its mistakes and weakest explanations on the evaluation set, the earlier rounds, the objectives, the budget and the models allowed."""
    config = versions[current - 1]
    report = {
        "current_version": config.model_dump(),
        "scores": {set_name: average(current, set_name) for set_name in SETS},
        "evaluation_set": mistakes(current),
        "earlier_rounds": rounds,
        "objectives_on_the_held_out_set": OBJECTIVES,
        "noise": NOISE,
        "tolerance": TOLERANCE,
        "rule": "A new version is kept when it meets the objectives, at least one score improves by more than the noise and none gets worse by more than the tolerance",
        "budget_left": round(BUDGET - spent, 4),
        "models_allowed": ALLOWED_MODELS,
        "policy_wordings_allowed": {
            name: {"characters": len(text), "description": description}
            for (name, text), description in zip(
                WORDINGS.items(),
                [
                    "Original full policy wording",
                    "Concise policy wording; section titles, no numbers",
                ],
                strict=True,
            )
        },
    }
    return json.dumps(report, indent=2, default=str)

In [ ]:
@tool(parse_docstring=True)
def propose_change(
    instructions: str,
    model: str,
    reasoning_effort: str,
    policy_wording: Literal["full", "short"],
    reason: str,
) -> str:
    """Propose a new version of the claim agent's configuration. A human reviews it before it's saved.

    Args:
        instructions: The claim agent's complete instructions
        model: One of the models allowed
        reasoning_effort: A reasoning level allowed for that model
        policy_wording: full for the original wording, short for the concise wording
        reason: Why this change should improve the scores
    """
    if reasoning_effort not in ALLOWED_MODELS.get(model, {}).get(
        "reasoning_efforts", []
    ):
        return f"{model} at {reasoning_effort} isn't allowed. Choose from the models allowed."
    version = len(versions) + 1
    versions.append(
        Configuration(
            version=version,
            instructions=instructions,
            model=model,
            reasoning_effort=reasoning_effort,
            policy_wording=policy_wording,
            reason=reason,
        )
    )
    return f"Saved as version {version}. Run the evaluation next."

In [ ]:
@tool
def run_evaluation() -> str:
    """Run the newest version once on the evaluation set and twice on the held-out set, then keep it or roll it back."""
    global current
    config = versions[-1]
    if config.version == current or any(r["version"] == config.version for r in rounds):
        return "There's no new version to evaluate. Propose a change first."
    estimate = (
        3 * len(SETS["held-out"]) * max(row["cost_per_claim"] for row in scoreboard)
    )
    if spent + estimate > BUDGET:
        return f"Refused: the evaluation could cost about ${estimate:.2f} and ${BUDGET - spent:.2f} of the budget is left"
    before = average(current, "held-out")
    evaluate(config, "evaluation")
    evaluate(config, "held-out")
    evaluate(config, "held-out")
    after = average(config.version, "held-out")
    keep, why = keep_or_roll_back(after, before)
    rounds.append(
        {
            "round": len(rounds) + 1,
            "recipe_version": len(recipes),
            "version": config.version,
            "model": config.model,
            "reasoning_effort": config.reasoning_effort,
            "reason": config.reason,
            "policy_wording": config.policy_wording,
            "kept": keep,
            "why": why,
            "held_out_before": before,
            "held_out_after": after,
        }
    )
    if keep:
        current = config.version
    verdict = "Kept" if keep else "Rolled back"
    scores = {
        "held_out_before": before,
        "held_out_after": after,
        "evaluation_after": latest(config.version, "evaluation"),
    }
    return f"{verdict} version {config.version}: {why}. Version {current} is now the current version.\n{json.dumps(scores, indent=2)}"

In [ ]:
recipes = []


@tool(parse_docstring=True)
def propose_recipe(recipe: str, reason: str) -> str:
    """Propose a new version of your recipe for improving the claim agent. A human reviews it before it's saved.

    Args:
        recipe: The complete new recipe
        reason: What this round showed, and why the new recipe should lead to better changes
    """
    recipes.append(recipe)
    return f"Saved as recipe version {len(recipes)}."

In [ ]:
RECIPE = """1. Call read_results. Check objectives, allowed models and wording choices, earlier rounds and remaining budget.
2. In the first round, test short instead of full wording, keeping instructions, model and reasoning unchanged. In later rounds, use the evidence to choose one focused change; do not add claim-specific rules.
3. Call propose_change with the complete configuration, including policy_wording, and explain your hypothesis. Wait for human approval.
4. Call run_evaluation once. Respect its keep-or-rollback verdict; a cheaper prompt still has to meet the quality objectives.
5. Use this round's evidence to improve the recipe with propose_recipe. Preserve human approval, budget and evaluation rules, the policy wording choice, and this final recipe-improvement step.
"""
recipes.append(RECIPE)

In [ ]:
approval = HumanInTheLoopMiddleware(
    interrupt_on={
        "propose_change": {"allowed_decisions": ["approve", "reject"]},
        "propose_recipe": {"allowed_decisions": ["approve", "reject"]},
    }
)


def make_improver(recipe: str) -> CompiledStateGraph:
    """Start a fresh improver with the latest approved recipe."""
    improver_model = ChatOpenAI(
        model="openai/gpt-6-luna",
        base_url=OPENROUTER_BASE_URL,
        api_key=openrouter_api_key,
        reasoning_effort="medium",
        timeout=300,
        max_retries=5,
    )

    system_prompt = f"""You improve the configuration of UnderCover's claim agent, an AI agent that decides travel insurance claims.
The claim agent sees its instructions, the policy wording, the customer's policy record and the claim. It doesn't see the claims manual or the customer's history.
Follow your recipe one step at a time.

# Your recipe

{recipe}
"""
    return create_agent(
        model=improver_model,
        tools=[read_results, propose_change, run_evaluation, propose_recipe],
        system_prompt=system_prompt,
        middleware=[approval],
        checkpointer=InMemorySaver(),
    )

In [ ]:
def show_changed_lines(old: str, new: str) -> None:
    """Print the lines that differ between two texts."""
    for line in difflib.unified_diff(
        old.splitlines(), new.splitlines(), lineterm="", n=0
    ):
        if not line.startswith(("---", "+++", "@@")):
            print(line)


def show_proposal(result: dict) -> None:
    """Show the change the improver is waiting for us to approve."""
    if "__interrupt__" not in result:
        print("The improver finished without a proposal. Its last message:")
        print(result["messages"][-1].content)
        return
    request = result["__interrupt__"][0].value["action_requests"][0]
    args = request["args"]
    if request["name"] == "propose_change":
        old = versions[current - 1]
        print(
            f"Model: {old.model} at {old.reasoning_effort}, proposed {args['model']} at {args['reasoning_effort']}\n"
        )
        print(f"Policy wording: {old.policy_wording} -> {args['policy_wording']}\n")
        show_changed_lines(old.instructions, args["instructions"])
    else:
        print(f"Recipe version {len(recipes)}, proposed version {len(recipes) + 1}\n")
        show_changed_lines(recipes[-1], args["recipe"])
    print(f"\nReason: {args['reason']}")


def approve_pending(result: dict, tool_name: str) -> dict:
    """Approve only the single proposal the preceding cell displayed."""
    interrupts = result.get("__interrupt__", [])
    actions = [
        action for item in interrupts for action in item.value["action_requests"]
    ]
    if len(actions) != 1 or actions[0]["name"] != tool_name:
        raise ValueError(
            f"Expected one pending {tool_name} proposal; inspect the result before proceeding."
        )
    return improver.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}), round_config
    )

## Offline checks

These check prompt selection, the configuration/tool schema and the quality-versus-cost gate without credentials or model calls.

In [ ]:
baseline = versions[0]
short = baseline.model_copy(update={"policy_wording": "short"})
assert WORDINGS["full"] != WORDINGS["short"]
assert len(WORDINGS["short"]) < len(WORDINGS["full"])
assert claim_system_prompt(baseline) == f"{INSTRUCTIONS}\n{WORDINGS['full']}\n"
assert claim_system_prompt(short) == f"{INSTRUCTIONS}\n{WORDINGS['short']}\n"
try:
    Configuration(**{**baseline.model_dump(), "policy_wording": "unknown"})
except ValidationError:
    pass
else:
    raise AssertionError("Unknown wording must be rejected")
schema = propose_change.get_input_schema().model_json_schema()
assert schema["properties"]["policy_wording"]["enum"] == ["full", "short"]
before = {
    "accuracy": 0.8,
    "grade": 2.0,
    "cost_per_claim": 0.001,
    "seconds_per_claim": 10.0,
}
cheaper = {**before, "cost_per_claim": 0.0007}
assert keep_or_roll_back(cheaper, before)[0]
assert not keep_or_roll_back({**cheaper, "accuracy": 0.5}, before)[0]
assert not keep_or_roll_back({**cheaper, "grade": 1.5}, before)[0]
assert not keep_or_roll_back({**before, "cost_per_claim": 0.0009}, before)[0]
print("Offline checks passed. No model calls made.")

## Live baseline

Set `RUN_LIVE = True` when ready to spend API credits. Root `.env` supplies OpenRouter and Langfuse keys. The baseline uses full wording: one evaluation run and two held-out runs. `TINY = True` above reduces each set to five claims, useful only as a smoke test.

The lab's $0.50 guard estimates evaluation costs from earlier runs; it is not a hard spending cap and excludes improver calls. Prompt caching, output length and repeated tool calls can offset input savings.

In [ ]:
RUN_LIVE = False
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
if RUN_LIVE:
    load_dotenv(ROOT / ".env", override=True)
    openrouter_api_key = os.environ["OPENROUTER_API_KEY"]
    openrouter_decisions = OpenRouter(api_key=openrouter_api_key, timeout_ms=60_000)
    langfuse = get_client()
    langfuse_handler = CallbackHandler()
    if not langfuse.auth_check():
        raise RuntimeError("Check the Langfuse settings in the root .env")
    for set_name in ["evaluation", "held-out", "held-out"]:
        evaluate(versions[0], set_name)
    display(pd.DataFrame(scoreboard).round(4))

## One improvement round

Run these three cells in order, inspecting each proposal before setting its approval flag to `True`. To reject a proposal instead, resume the same improver and `round_config` with `Command(resume={"decisions": [{"type": "reject", "message": "your reason"}]})`, then inspect the new proposal.

After the recipe is approved, repeat the three cells for another round. Each round starts a new improver from the latest recipe and current accepted configuration. Never rerun the baseline during an active round.

In [ ]:
if RUN_LIVE:
    round_number = len(rounds) + 1
    improver = make_improver(recipes[-1])
    round_config = {
        "configurable": {"thread_id": f"round-{round_number}"},
        "callbacks": [langfuse_handler],
        "run_name": f"Policy wording round {round_number}",
        "metadata": {
            "langfuse_session_id": f"Policy wording {datetime.now():%Y-%m-%d %H:%M}"
        },
    }
    result = improver.invoke(
        {
            "messages": [
                {"role": "user", "content": "Carry out one round of your recipe."}
            ]
        },
        round_config,
    )
    show_proposal(result)

In [ ]:
APPROVE_CHANGE = False
if RUN_LIVE and APPROVE_CHANGE:
    result = approve_pending(result, "propose_change")
    show_proposal(result)
    display(
        pd.DataFrame(rounds)[
            ["round", "recipe_version", "version", "policy_wording", "kept", "why"]
        ]
    )

In [ ]:
APPROVE_RECIPE = False
if RUN_LIVE and APPROVE_RECIPE:
    result = approve_pending(result, "propose_recipe")
    show_proposal(result)

## Compare measured results

The table averages repeated runs. Cost is the claim agent's actual reported API cost, not a token-count estimate. `cost_change_pct` is relative to version 1; negative means cheaper. Judge cost is included in evaluation spend, separately from the per-claim metric. A rejected candidate stays in the history; only `current` is adopted. If the short version loses quality or saves less than the noise threshold, report that result rather than claiming improvement.

In [ ]:
if scoreboard:
    summary = (
        pd.DataFrame(scoreboard)
        .groupby(["version", "wording", "set"], as_index=False)
        .mean(numeric_only=True)
    )
    baseline_cost = summary[summary["version"] == 1].set_index("set")["cost_per_claim"]
    summary["cost_change_pct"] = 100 * (
        summary["cost_per_claim"] / summary["set"].map(baseline_cost) - 1
    )
    display(summary.round(4))
    print(
        f"Current version: {current}, wording: {versions[current - 1].policy_wording}"
    )
    print(f"Evaluations including judge: ${spent:.4f} of ${BUDGET:.2f} budget")
    print(
        f"Recipes: {len(recipes)}. Clear all notebook outputs before submitting a PR."
    )